In [1]:
tabla ='qtioo10'
col_tabla = 'solopefec'
essi='essi_dat_cqx005_etl'
col_essi='sol_fec'
dat= "dat_ceqx005_essi"
col_dat='sol_fec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [3]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [4]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [5]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [6]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [7]:

fecha= '2023-05-01'

In [8]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

cps= pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
cps["cod_cps"]=cps["cod_cps"].str.strip()

areas = pd.read_sql_query(f"SELECT id_area,cod_are FROM dim_areas", con=connection2)
areas['cod_are']=areas['cod_are'].str.strip()

servicios = pd.read_sql_query(f"SELECT id_serv,cod_ser FROM dim_servicios", con=connection2)
servicios['cod_ser']=servicios['cod_ser'].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)


In [9]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [10]:
base1.shape

(46615, 17)

In [11]:
#Eliminamos las columnas que no se usarán aquí: las descripciones previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'des_are', 'des_ser']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [12]:
base1.shape

(46615, 13)

In [13]:
inicioProceso=time.time()

In [14]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [15]:
base1.shape

(46615, 13)

In [16]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46615 entries, 0 to 46614
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ori_cas  46615 non-null  object 
 1   cod_cas  46615 non-null  object 
 2   cod_red  46615 non-null  object 
 3   sol_num  46615 non-null  float64
 4   sec_num  46615 non-null  float64
 5   cod_cps  46615 non-null  object 
 6   cps_ori  0 non-null      object 
 7   cps_cas  0 non-null      object 
 8   cps_are  0 non-null      object 
 9   cps_ser  0 non-null      object 
 10  sol_fec  46615 non-null  object 
 11  act_med  46615 non-null  float64
 12  pac_sec  46615 non-null  float64
dtypes: float64(4), object(9)
memory usage: 4.6+ MB


In [17]:
control_a.append(base1.shape[0])

In [18]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(46615, 14)

In [19]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [20]:
oricas=oricas.rename(columns={"ori_cas":"cps_ori","id_oricas":"id_cpsori"})
base1['cps_ori']=base1['cps_ori'].str.strip()
base1["cps_ori"]=base1["cps_ori"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='inner',on='cps_ori')
base1=pd.merge(base1,oricas,how='left',on='cps_ori')
base1.shape

(46615, 14)

In [21]:
merge.shape #Merge se baja todo porque no hay datos

(0, 14)

In [22]:
log_falla('id_cpsori', 'cps_ori', True)
log_control('dim_oricas')
base1=base1.drop('cps_ori',axis=1)

In [23]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='inner',on='cod_cas')
base1=pd.merge(base1,cas,how='left',on='cod_cas')
base1.shape

(46615, 14)

In [24]:
merge.shape

(46615, 14)

In [25]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [26]:
cas=cas.rename(columns={"cod_cas":"cps_cas","id_cas":"id_cpscas"})
base1['cps_cas']=base1['cps_cas'].str.strip()
base1["cps_cas"]=base1["cps_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='inner',on='cps_cas')
base1=pd.merge(base1,cas,how='left',on='cps_cas')
base1.shape

(46615, 14)

In [27]:
merge.shape

(0, 14)

In [28]:
log_falla('id_cpscas', 'cps_cas', True)
log_control('dim_cas')
base1=base1.drop('cps_cas',axis=1)

In [29]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='inner',on='cod_red')
base1=pd.merge(base1,red,how='left',on='cod_red')
base1.shape

(46615, 14)

In [30]:
merge.shape

(46615, 14)

In [31]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [32]:
base1['cod_cps']=base1['cod_cps'].str.strip()
base1["cod_cps"]=base1["cod_cps"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cps,how='inner',on='cod_cps')
base1=pd.merge(base1,cps,how='left',on='cod_cps')
base1.shape

(46615, 14)

In [33]:
merge.shape #Merge pierde todo

(46615, 14)

In [34]:
log_falla('id_cps', 'cod_cps', True)
log_control('dim_cps')
base1=base1.drop('cod_cps',axis=1)

In [35]:
areas=areas.rename(columns={"cod_are": "cps_are","id_area":"id_cpsare"})
base1['cps_are']=base1['cps_are'].str.strip()
base1["cps_are"]=base1["cps_are"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,areas,how='inner',on='cps_are')
base1=pd.merge(base1,areas,how='left',on='cps_are')
base1.shape

(46615, 14)

In [36]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sol_num    46615 non-null  float64
 1   sec_num    46615 non-null  float64
 2   cps_are    0 non-null      object 
 3   cps_ser    0 non-null      object 
 4   sol_fec    46615 non-null  object 
 5   act_med    46615 non-null  float64
 6   pac_sec    46615 non-null  float64
 7   id_oricas  46615 non-null  int64  
 8   id_cpsori  0 non-null      float64
 9   id_cas     46615 non-null  int64  
 10  id_cpscas  0 non-null      float64
 11  id_red     46615 non-null  int64  
 12  id_cps     46615 non-null  int64  
 13  id_cpsare  0 non-null      float64
dtypes: float64(7), int64(4), object(3)
memory usage: 5.3+ MB


In [37]:
log_falla('id_cpsare', 'cps_are', True)
log_control('dim_areas')
base1 = base1.drop("cps_are",axis=1)

In [38]:
servicios=servicios.rename(columns={"cod_ser": "cps_ser","id_serv":"id_cpsser"})
base1['cps_ser']=base1['cps_ser'].str.strip()
base1["cps_ser"]=base1["cps_ser"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,servicios,how='inner',on='cps_ser')
base1=pd.merge(base1,servicios,how='left',on='cps_ser')
base1.shape

(46615, 14)

In [39]:
merge.shape

(0, 14)

In [40]:
log_falla('id_cpsser','cps_ser', True)
log_control('dim_servicios')
base1 = base1.drop("cps_ser",axis=1)

In [41]:
pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(46615, 14)

In [42]:
merge.shape

(45775, 14)

In [43]:
log_falla('id_pacsec', 'pac_sec', True)
base1=base1.drop('pac_sec',axis=1)
dim.append('dim_pacsec')
control_d.append(base1.shape[0])

In [44]:
base1['sol_fec_temp'] = base1['sol_fec'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"sol_fec_temp"})
tiempo["sol_fec_temp"] = tiempo['sol_fec_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='sol_fec_temp')
base1 = pd.merge(base1, tiempo, how='left', on='sol_fec_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_sol","dt_ano":"id_ano_sol","dt_mes":"id_mes_sol",
                            "dt_dia":"id_dia_sol","dt_dia_sem":"id_diasem_sol","dt_sem":"id_sem_sol"})
base1=base1.drop("sol_fec_temp",axis=1)
base1.shape

(46615, 19)

In [45]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sol_num        46615 non-null  float64
 1   sec_num        46615 non-null  float64
 2   sol_fec        46615 non-null  object 
 3   act_med        46615 non-null  float64
 4   id_oricas      46615 non-null  int64  
 5   id_cpsori      0 non-null      float64
 6   id_cas         46615 non-null  int64  
 7   id_cpscas      0 non-null      float64
 8   id_red         46615 non-null  int64  
 9   id_cps         46615 non-null  int64  
 10  id_cpsare      0 non-null      float64
 11  id_cpsser      0 non-null      float64
 12  id_pacsec      45775 non-null  float64
 13  id_time_sol    46614 non-null  float64
 14  id_mes_sol     46614 non-null  float64
 15  id_dia_sol     46614 non-null  float64
 16  id_diasem_sol  46614 non-null  float64
 17  id_sem_sol     46614 non-null  float64
 18  id_ano

In [46]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [47]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [48]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [49]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [50]:
base

,id_cpsori,id_dia_sol,id_time_sol,id_cpsare,id_pacsec,id_cpscas,id_mes_sol,id_cpsser,sec_num,id_diasem_sol,id_red,id_cps,act_med,sol_num,id_cas,id_oricas,sol_fec,id_sem_sol,id_ano_sol
0,NaN,NaN,NaN,NaN,39641913.0,NaN,NaN,NaN,1.0,NaN,3,83782,6799154.0,5182.0,124,1,2029-07-10,NaN,NaN
1,NaN,2.0,20230602.0,NaN,46964915.0,NaN,6.0,NaN,1.0,5.0,3,84341,11250874.0,63927.0,565,1,2023-06-02,1.0,2023.0
2,NaN,2.0,20230602.0,NaN,46964915.0,NaN,6.0,NaN,1.0,5.0,3,78261,11250874.0,63927.0,565,1,2023-06-02,1.0,2023.0
3,NaN,5.0,20250805.0,NaN,45575810.0,NaN,8.0,NaN,1.0,2.0,7,77523,1504924.0,13541.0,310,1,2025-08-05,2.0,2025.0
4,NaN,2.0,20230902.0,NaN,56422139.0,NaN,9.0,NaN,1.0,6.0,3,81059,11039911.0,72546.0,565,1,2023-09-02,2.0,2023.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46610,NaN,19.0,20230619.0,NaN,55692290.0,NaN,6.0,NaN,1.0,1.0,3,77588,12003580.0,29950.0,99,1,2023-06-19,1.0,2023.0
46611,NaN,26.0,20230626.0,NaN,55051470.0,NaN,6.0,NaN,1.0,1.0,3,79119,13656916.0,115236.0,565,1,2023-06-26,1.0,2023.0
46612,NaN,25.0,20230625.0,NaN,38823107.0,NaN,6.0,NaN,1.0,7.0,2,78526,8728178.0,130307.0,588,1,2023-06-25,1.0,2023.0
46613,NaN,24.0,20230524.0,NaN,54384131.0,NaN,5.0,NaN,1.0,3.0,9,86590,2144583.0,38358.0,265,1,2023-05-24,1.0,2023.0


In [51]:
base.columns

Index(['id_cpsori', 'id_dia_sol', 'id_time_sol', 'id_cpsare', 'id_pacsec',
       'id_cpscas', 'id_mes_sol', 'id_cpsser', 'sec_num', 'id_diasem_sol',
       'id_red', 'id_cps', 'act_med', 'sol_num', 'id_cas', 'id_oricas',
       'sol_fec', 'id_sem_sol', 'id_ano_sol'],
      dtype='object')

In [52]:
base.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 46615 entries, 0 to 46614
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_cpsori      0 non-null      float64
 1   id_dia_sol     46614 non-null  float64
 2   id_time_sol    46614 non-null  float64
 3   id_cpsare      0 non-null      float64
 4   id_pacsec      45775 non-null  float64
 5   id_cpscas      0 non-null      float64
 6   id_mes_sol     46614 non-null  float64
 7   id_cpsser      0 non-null      float64
 8   sec_num        46615 non-null  float64
 9   id_diasem_sol  46614 non-null  float64
 10  id_red         46615 non-null  int64  
 11  id_cps         46615 non-null  int64  
 12  act_med        46615 non-null  float64
 13  sol_num        46615 non-null  float64
 14  id_cas         46615 non-null  int64  
 15  id_oricas      46615 non-null  int64  
 16  sol_fec        46615 non-null  object 
 17  id_sem_sol     46614 non-null  float64
 18  id_ano

In [53]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

4615

In [54]:
fecha_fin = "2024-12-31"

In [55]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [56]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_oricas,2023-05-01,2024-12-31,46615,46615,0,0,id_oricas
1,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_oricas,2023-05-01,2024-12-31,46615,46615,0,0,id_cpsori
2,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_cas,2023-05-01,2024-12-31,46615,46615,0,0,id_cas
3,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_cas,2023-05-01,2024-12-31,46615,46615,0,0,id_cpscas
4,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_red,2023-05-01,2024-12-31,46615,46615,0,0,id_red
5,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_cps,2023-05-01,2024-12-31,46615,46615,0,0,id_cps
6,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_areas,2023-05-01,2024-12-31,46615,46615,0,0,id_cpsare
7,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_servicios,2023-05-01,2024-12-31,46615,46615,0,0,id_cpsser
8,13,DAT,dat_ceqx005_essi,2023-06-27,16:41,16:43,dim_pacsec,2023-05-01,2024-12-31,46615,46615,0,0,id_pacsec


In [57]:
tabla_logs.to_sql(name=f'logs', con=connection4, if_exists='append', index=False,chunksize=10000)

ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permiso denegado a la tabla logs

[SQL: INSERT INTO logs (codigo, proceso, dat, fecha_ejecucion, hora_inicio, hora_fin, dim, fecha_ini, fecha_ter, esperado, obtenido, faltante, falla, id_afectado) VALUES (%(codigo)s, %(proceso)s, %(dat)s, %(fecha_ejecucion)s, %(hora_inicio)s, %(hora_fin)s, %(dim)s, %(fecha_ini)s, %(fecha_ter)s, %(esperado)s, %(obtenido)s, %(faltante)s, %(falla)s, %(id_afectado)s)]
[parameters: ({'codigo': 13, 'proceso': 'DAT', 'dat': 'dat_ceqx005_essi', 'fecha_ejecucion': '2023-06-27', 'hora_inicio': '16:41', 'hora_fin': '16:43', 'dim': 'dim_oricas', 'fecha_ini': '2023-05-01', 'fecha_ter': '2024-12-31', 'esperado': 46615, 'obtenido': 46615, 'faltante': 0, 'falla': 0, 'id_afectado': 'id_oricas'}, {'codigo': 13, 'proceso': 'DAT', 'dat': 'dat_ceqx005_essi', 'fecha_ejecucion': '2023-06-27', 'hora_inicio': '16:41', 'hora_fin': '16:43', 'dim': 'dim_oricas', 'fecha_ini': '2023-05-01', 'fecha_ter': '2024-12-31', 'espera

In [ ]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 276.906 segundos


In [ ]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [ ]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()